# ROBOP Analysis Notebook

Exploratory analysis of pose estimation results.  
Utilities live in `analysis_utils.py`; plot helpers in `analyze.py` - import, don't copy.

**Sections**
1. Setup & data loading
2. Summary metrics
3. Failure analysis + oracle gap
4. Error decomposition (per-axis translation + rotation)
5. Signal correlations (Spearman ρ vs ADD)
6. AUROC per signal
7. Level 1 gate - pre-PnP signal analysis (`conf_50`, `conf_count_50`, `anc_r`)
8. Level 2 gate - IoU Pareto frontier (PnP-success frames)
9. Cross-dataset comparison

---
## 1. Setup & data loading

In [ ]:
import sys
import importlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import spearmanr

# Make analysis_utils importable from anywhere
ANALYSIS_DIR = Path(".").resolve()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

from analysis_utils import (
    load_results, detect_dataset, is_nemo,
    extract_signals, error_decomposition,
    naive_score, compute_auroc, pareto_curve, operating_point,
    print_header, print_table, spearman_table,
)
import analysis_utils as _au
importlib.reload(_au)

# Re-use plot helpers from analyze.py - reload so edits take effect without kernel restart
import analyze as _az
importlib.reload(_az)

%matplotlib inline
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "figure.dpi": 130,
})
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")

# Shared colour palette (mirrors analyze._C)
C = {
    "blue":   "#4C72B0",
    "orange": "#DD8452",
    "green":  "#55A868",
    "red":    "#C44E52",
    "purple": "#8172B2",
    "grey":   "#8C8C8C",
}

FAILURE_THR = 100.0  # mm - change here to affect all sections

In [ ]:
# -- Result files -------------------------------------------------------------
# Map short plot labels to evaluation result paths.
RESULT_FILES = {}

# -- Single active file for Sections 2-8 --------------------------------------
# Set to any key from RESULT_FILES, or leave as-is - auto-falls back to the
# first loaded key if the value doesn't match.
ACTIVE = None   # None = always pick first loaded file

# -- External baseline AUC values for Pareto reference lines ------------------
# Fill in once you have confirmed numbers from the papers.
BASELINE_AUC = {
    "panda_nemo":  None,   # e.g. 0.812  (CtRNet ADD AUC@100mm)
    "baxter_nemo": None,   # e.g. 0.745  (CtRNet ADD AUC@100mm)
    "craves_nemo": None,   # e.g. ...    (CRAVES paper)
}

# -- Load ---------------------------------------------------------------------
datasets = {}
for label, path in RESULT_FILES.items():
    summary, queries = load_results(path)
    signals  = extract_signals(queries)
    dataset  = detect_dataset(summary)
    nemo     = is_nemo(queries)
    datasets[label] = dict(
        summary=summary, queries=queries,
        signals=signals, dataset=dataset, nemo=nemo,
    )
    print(f"{label:20s}  N={len(queries):4d}  dataset={dataset}  nemo={nemo}")

if not datasets:
    print("No result files loaded - fill in RESULT_FILES above.")
else:
    if ACTIVE is None or ACTIVE not in datasets:
        if ACTIVE is not None:
            print(f"  WARNING: ACTIVE='{ACTIVE}' not found in loaded files.")
        ACTIVE = next(iter(datasets))
        print(f"  Using first loaded file as active: '{ACTIVE}'")
    d       = datasets[ACTIVE]
    SUM     = d["summary"]
    Q       = d["queries"]
    SIG     = d["signals"]
    DS      = d["dataset"]
    IS_NEMO = d["nemo"]
    print(f"\nActive: '{ACTIVE}'  ({DS}, {'NeMO' if IS_NEMO else 'DINOv2'}, N={len(Q)})")

---
## 2. Summary metrics

In [ ]:
_az.section_summary(SUM, DS, IS_NEMO)

---
## 3. Failure analysis + oracle gap

In [ ]:
_az.section_failures(Q, SIG, FAILURE_THR)

In [ ]:
_az._setup_style()
_az._plot_add_histogram(SIG, DS, FAILURE_THR, out_dir=None, plt=plt)

---
## 4. Error decomposition

In [ ]:
_az.section_error_decomposition(Q, DS)

In [ ]:
_az._plot_error_decomposition(Q, DS, out_dir=None, plt=plt, component="translation")

In [ ]:
# Prints a clear message for Baxter (no rotation GT); produces radar+lollipop for panda/craves
_az._plot_error_decomposition(Q, DS, out_dir=None, plt=plt, component="rotation")

---
## 5. Signal correlations  *(NeMO only)*

In [ ]:
if IS_NEMO:
    _az.section_signal_correlations(SIG, FAILURE_THR)
else:
    print("DINOv2 result - no NeMO confidence signals.")

In [ ]:
if IS_NEMO:
    _az._plot_signal_correlations(SIG, out_dir=None, plt=plt)

---
## 6. AUROC per signal  *(NeMO only)*

> **Caveat:** `conf_50` and `anc_r` are non-NaN for all N frames
> (including PnP failures); `iou_*`, `reproj`, `inlier` are non-NaN only for PnP-success
> frames. Direct AUROC comparisons across these groups are misleading - see Section 7/8
> for the split analysis.

In [ ]:
if IS_NEMO:
    _az.section_auroc(SIG, FAILURE_THR)
else:
    print("DINOv2 result - no NeMO confidence signals.")

In [ ]:
if IS_NEMO:
    _az._plot_auroc_bars(SIG, FAILURE_THR, out_dir=None, plt=plt)

---
## 7. Level 1 gate - pre-PnP signal analysis  *(NeMO only)*

`conf_50`, `conf_count_50`, and `anc_r` are available **before PnP runs** - they
cover all frames including PnP failures.  
Question: can they predict PnP failure reliably enough to gate?

`conf_count_50` is the most direct predictor: PnP failure fires when pixel count
below `min_correspondences` (default 32), so `conf_count_50` measures exactly that.

In [ ]:
if IS_NEMO:
    _az.section_level1_gating(SIG)

In [ ]:
if IS_NEMO:
    _az._plot_level1_distributions(SIG, out_dir=None, plt=plt)

In [ ]:
# -- Threshold sweep on conf_count_50  (Level 1 gate) -------------------------
# How many PnP failures could be caught at various count thresholds,
# and how many good frames would be wrongly rejected?

if IS_NEMO:
    pnp_failed    = SIG["pnp_failed"]
    conf_count    = SIG["conf_count_50"]
    valid_count   = ~np.isnan(conf_count)

    thresholds = np.arange(0, 300, 10)
    tpr_list, fpr_list, precision_list = [], [], []

    for t in thresholds:
        # gate fires (predict failure) when count < t
        predicted_fail = conf_count < t
        tp = int(np.sum( pnp_failed &  predicted_fail & valid_count))
        fn = int(np.sum( pnp_failed & ~predicted_fail & valid_count))
        fp = int(np.sum(~pnp_failed &  predicted_fail & valid_count))
        tn = int(np.sum(~pnp_failed & ~predicted_fail & valid_count))
        tpr_list.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
        fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else np.nan)
        prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        precision_list.append(prec)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(thresholds, tpr_list, color=C["green"],  lw=2, label="TPR (PnP failures caught)")
    axes[0].plot(thresholds, fpr_list, color=C["red"],    lw=2, label="FPR (good frames rejected)")
    axes[0].plot(thresholds, precision_list, color=C["blue"], lw=2, linestyle="--", label="Precision")
    axes[0].set_xlabel("conf_count_50 threshold")
    axes[0].set_ylabel("Rate")
    axes[0].set_title("Level 1 Gate: TPR / FPR / Precision vs Threshold")
    axes[0].legend()
    axes[0].set_ylim(-0.02, 1.05)

    axes[1].plot(fpr_list, tpr_list, color=C["blue"], lw=2)
    axes[1].plot([0, 1], [0, 1], color=C["grey"], lw=1, linestyle="--")
    axes[1].set_xlabel("FPR (good frames wrongly rejected)")
    axes[1].set_ylabel("TPR (PnP failures caught)")
    axes[1].set_title("ROC - conf_count_50 predicting PnP failure")
    for t, fpr, tpr in zip(thresholds[::3], fpr_list[::3], tpr_list[::3]):
        if not (np.isnan(fpr) or np.isnan(tpr)):
            axes[1].annotate(str(int(t)), (fpr, tpr), fontsize=7, color=C["grey"])

    fig.tight_layout()
    plt.show()

---
## 8. Level 2 gate - IoU Pareto frontier  *(NeMO only)*

The two Pareto curves operate on different frame subsets:

| Curve | Frame subset | Gate signal |
|-------|-------------|-------------|
| naive_score | all frames with valid score (incl. PnP failures via conf_50/anc_r) | equal-weight composite |
| iou_modal   | PnP-success frames only (x-axis rescaled to all frames) | IoU modal vs render |

In [ ]:
if IS_NEMO:
    _az.section_gating(SIG, FAILURE_THR)
    _az.section_iou_analysis(SIG, FAILURE_THR)

In [ ]:
if IS_NEMO:
    _az._plot_iou_distributions(SIG, FAILURE_THR, out_dir=None, plt=plt)

In [ ]:
# -- Extended Pareto with IoU curve + external baseline reference --------------
if IS_NEMO:
    add       = SIG["add_m_mm"]
    total_n   = len(add)

    score_naive  = naive_score(SIG)
    valid_naive  = ~np.isnan(score_naive)
    iou_score    = SIG["iou_modal"]
    valid_iou    = ~np.isnan(iou_score)

    auc_all      = float(np.mean([np.mean(add < t) for t in range(100)]))
    n_pnp_succ   = int(valid_iou.sum())

    fig, ax = plt.subplots(figsize=(10, 5.5))

    # Unfiltered baseline
    ax.axhline(auc_all, color=C["red"], lw=1.8, linestyle="--",
               label=f"Unfiltered (all {total_n} frames)  AUC={auc_all:.3f}")

    # External baseline (CtRNet / DREAM)
    ext = BASELINE_AUC.get(ACTIVE)
    if ext is not None:
        ax.axhline(ext, color=C["grey"], lw=1.8, linestyle=":",
                   label=f"CtRNet/DREAM baseline  AUC={ext:.3f}")

    # Level 1 + naive_score Pareto (all frames)
    if valid_naive.sum() >= 20:
        cov_n, auc_n = pareto_curve(score_naive[valid_naive], add[valid_naive], FAILURE_THR)
        ax.plot(cov_n * 100, auc_n, color=C["blue"], lw=2.5,
                label=f"naive_score gate (N={valid_naive.sum()}, incl. PnP failures)")
        for c_tgt, marker in zip([0.9, 0.8, 0.7], ["o", "s", "^"]):
            op = operating_point(score_naive[valid_naive], add[valid_naive],
                                 c_tgt, FAILURE_THR, total_n=total_n)
            ax.scatter([op["actual_coverage"] * 100], [op["auc_filtered"]],
                       s=90, zorder=6, marker=marker, color=C["blue"],
                       label=f"  @ {c_tgt*100:.0f}% cov -> AUC={op['auc_filtered']:.3f}")

    # Level 2 - iou_modal Pareto (PnP-success frames only)
    if valid_iou.sum() >= 20:
        auc_pnp_base = float(np.mean([np.mean(add[valid_iou] < t) for t in range(100)]))
        cov_iou, auc_iou = pareto_curve(iou_score[valid_iou], add[valid_iou], FAILURE_THR)
        # Rescale x-axis so both curves share the same denominator (all frames)
        cov_iou_global = cov_iou * n_pnp_succ / total_n
        ax.plot(cov_iou_global * 100, auc_iou, color=C["green"], lw=2.5,
                label=f"iou_modal gate (N={n_pnp_succ} PnP-success frames)")
        ax.axhline(auc_pnp_base, color=C["green"], lw=1.2, linestyle=":",
                   alpha=0.7, label=f"PnP-success baseline  AUC={auc_pnp_base:.3f}")
        for c_tgt, marker in zip([0.9, 0.8, 0.7], ["o", "s", "^"]):
            op = operating_point(iou_score[valid_iou], add[valid_iou],
                                 c_tgt, FAILURE_THR, total_n=n_pnp_succ)
            ax.scatter([op["actual_coverage"] * n_pnp_succ / total_n * 100],
                       [op["auc_filtered"]],
                       s=90, zorder=6, marker=marker, color=C["green"])

    ax.set_xlabel("Coverage (% of all frames)")
    ax.set_ylabel(f"ADD AUC @ {FAILURE_THR:.0f} mm")
    ax.set_title(f"Pareto Frontier - {ACTIVE}\n"
                 "(green = Level 2 IoU gate on PnP-success frames; x-axis normalised to all frames)")
    ax.legend(loc="lower left", fontsize=8.5)
    ax.set_xlim(0, 101)
    fig.tight_layout()
    plt.show()

---
## 9. Cross-dataset comparison  *(NeMO only)*

Checks whether AUROC values for the key signals hold across panda, baxter, and craves.  
Requires all three result files to be loaded in `RESULT_FILES`.

In [ ]:
# -- Cross-dataset AUROC table -------------------------------------------------
CROSS_SIGNALS = ["conf_50", "conf_count_50", "reproj", "inlier",
                 "anc_r", "iou_modal", "iou_amodal", "naive_combined"]
INVERT = {"reproj", "anc_r"}

nemo_datasets = {k: v for k, v in datasets.items() if v["nemo"]}
if len(nemo_datasets) < 2:
    print("Load at least two NeMO result files to enable cross-dataset comparison.")
else:
    col_labels = list(nemo_datasets.keys())
    header = ["Signal"] + col_labels
    rows = []
    for sig in CROSS_SIGNALS:
        row = [sig]
        for label, dset in nemo_datasets.items():
            sigs  = dset["signals"]
            add   = sigs["add_m_mm"]
            labels_bin = (add < FAILURE_THR).astype(int)
            if sig == "naive_combined":
                arr = naive_score(sigs)
                scores = arr
            else:
                arr = sigs.get(sig)
                if arr is None:
                    row.append("n/a")
                    continue
                scores = -arr if sig in INVERT else arr
            valid = ~np.isnan(scores)
            if (valid.sum() < 10 or labels_bin[valid].sum() == 0
                    or (1 - labels_bin[valid]).sum() == 0):
                row.append("n/a")
            else:
                auroc = compute_auroc(scores[valid], labels_bin[valid])
                row.append(f"{auroc:.3f}")
        rows.append(row)
    print_table(header, rows, col_width=16)

In [ ]:
# -- Cross-dataset AUROC bar chart ---------------------------------------------
if len(nemo_datasets) >= 2:
    ds_labels  = list(nemo_datasets.keys())
    ds_colors  = [C["blue"], C["orange"], C["green"], C["purple"]]
    key_signals = ["conf_count_50", "iou_modal", "naive_combined", "reproj", "inlier", "anc_r"]

    x    = np.arange(len(key_signals))
    width = 0.8 / max(len(ds_labels), 1)

    fig, ax = plt.subplots(figsize=(11, 5))
    for i, (label, dset) in enumerate(nemo_datasets.items()):
        sigs   = dset["signals"]
        add    = sigs["add_m_mm"]
        lbl_b  = (add < FAILURE_THR).astype(int)
        aurocs = []
        for sig in key_signals:
            if sig == "naive_combined":
                arr    = naive_score(sigs)
                scores = arr
            else:
                arr    = sigs.get(sig, np.full(len(add), np.nan))
                scores = -arr if sig in INVERT else arr
            valid = ~np.isnan(scores)
            if (valid.sum() < 10 or lbl_b[valid].sum() == 0
                    or (1 - lbl_b[valid]).sum() == 0):
                aurocs.append(np.nan)
            else:
                aurocs.append(compute_auroc(scores[valid], lbl_b[valid]))
        offsets = (i - (len(ds_labels) - 1) / 2) * width
        bars = ax.bar(x + offsets, aurocs, width=width * 0.9,
                      color=ds_colors[i % len(ds_colors)], label=label, alpha=0.85)

    ax.axhline(0.5, color="black", lw=1.2, linestyle="--", label="Random (0.5)")
    ax.set_xticks(x)
    ax.set_xticklabels(key_signals, rotation=15, ha="right")
    ax.set_ylabel(f"AUROC  (ADD < {FAILURE_THR:.0f}mm)")
    ax.set_title("Cross-Dataset AUROC Comparison")
    ax.set_ylim(0.4, 1.02)
    ax.legend()
    fig.tight_layout()
    plt.show()

---
## 10. Scratch / exploratory cells

Add new ideas here before promoting them to `analysis_utils.py` / `analyze.py`.